# Algoritmos de búsqueda de raíces

In [1]:
# Librerías utilizadas en este capítulo
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar, fsolve
from pathlib import Path
from IPython.display import HTML

plt.rcParams.update({'font.size': 10, 'figure.dpi': 200})

## Introducción
Consideremos el problema de caída de presión $\Delta P$ al mover un fluido con densidad $\rho$ y viscosidad cinemática $\nu$, a través de una tubería de largo $L$.

<img src="./images/presure_drop.png" width="300" align= center>

Para un fluido que fluye a una velocidad $V$, la caída de presión está dada por:

\begin{equation*}
\frac{\Delta P}{\rho g} =f_c \frac{L}{D}\frac{V^2}{2g}
\end{equation*}

donde $f_c$ es el factor de fricción.

Para determinar $f_c$ debemos resolver la ecuación de Colebrook:

\begin{equation*}
\frac{1}{\sqrt{f_c}} = -2.0\log\left(\frac{\varepsilon/D}{3.7} + \frac{2.51}{\mathrm{Re}\sqrt{f_c}}\right)
\end{equation*}

donde $\mathrm{Re} = \frac{VD}{\nu}$ es el número de Reynolds, y $\varepsilon/D$ la rugosidad relativa.

Sin embargo, esta ecuación no se puede resolver analíticamente. ¿Cómo resolvemos esta ecuación?

Llamamos raíces de una función $f(x)$ a los valores $x^*$ tales que $f(x^*) = 0$. 

Determinar $f_c$ a partir de la ecuación de Colebrook es equivalente a encontrar las raíces de la función:

\begin{equation*}
f(x) = \frac{1}{\sqrt{x}} + 2.0\log\left(\frac{\varepsilon/D}{3.7} + \frac{2.51}{\mathrm{Re}\sqrt{x}}\right)
\end{equation*}

En esta unidad revisaremos los algoritmos de búsqueda de raíces, primero para una función de una incógnita y después para sistemas de varias.

## Búsqueda de raíces para una función escalar

Resolver una ecuación de una incógnita equivale a encontrar la raíz de una función escalar: el valor $x^*$ tal que $f(x^*) = 0$.

Los algoritmos para hacerlo se clasifican en tres tipos:

- **Intervalo cerrado**: iteran dentro de un intervalo $x\in[a,b]$ que ya contiene la raíz.
- **Intervalo abierto**: iteran desde un valor inicial $x_0$, sin acotar la búsqueda.
- **Combinados**: alternan entre ambos según convenga.

Veremos la bisección (cerrado), Newton-Raphson y la secante (abiertos), y el método de Brent (combinado).

### Método de la Bisección (intervalo acotado)

El método de la bisección es un método de intervalo acotado que se basa en el teorema del valor intermedio

> **Teorema del valor intermedio** para una función $f(x)$ continua en entre los puntos $a$ y $b$, tal que $f(a)f(b) < 0$, existe un valor $c$, $a<c<b$, tal que $f(c) = 0$.

<img src="./images/valor_intermedio.png" width="380" align= center>

Para un intervalo $x\in [a,b]$, tal que $f(a)f(b) < 0$, el método de la bisección consiste en acotar el intervalo evaluando el punto medio $f(m)$, con $m = \frac{a+b}{2}$.

- Si $f(m)f(a) < 0$ el nuevo intervalo es $x\in [a,m]$, de lo contrario, $x\in [m, b]$

El algoritmo continua acotando el intervalo hasta encontrar la raíz de $f(x)$.

<img src="./images/bisection.png" width="380" align= center>

#### Criterio de convergencia
Para un nuevo valor $x_{k+1}$ obtenido a partir de un valor $x_k$, definimos el **criterio de convergencia** en base a:

- **error absoluto**

\begin{equation*}
|x_{k+1} -x_{k}|
\end{equation*}

- **error relativo**:

\begin{equation*}
\frac{|x_{k+1} -x_{k}|}{|x_{k+1}|}
\end{equation*}

Es importante menciona que, en general, no existe una regla respecto al tipo de error que se debe usar como criterio de convergencia. **La recomendación es usar el error absoluto si se tiene conocimiento de la función.** Esto se debe a que, a veces, el error relativo puede imponer condiciones demasiado estrictas para la convergencia.

Creemos una función en python para calcular raíces por medio del método de la bisección

In [2]:
import numpy as np
def bisection(f, bracket, abs_tol = 1E-5, max_iter = 100):
    '''
    Método de la bisección para encontrar la raíz de f(x)

    Parameters
        - f: callable
            función de búsqueda de raíz, f(x)

        - bracket: tupple
            intervalo de búsqueda (a, b)

        - abs_tol: float (optional)
            tolerancia al error absoluto (1E-5 por defecto)

        - max_iter: int (opcional)
            número máximo de iteraciones (100 por defecto)

    Return
        - x_sol: float
            raíz de f(x)
    '''
    a, b = bracket # extraemos los valores del intervalo

    # verificamos si el intervalo [a,b] satisface el teorema del valor intermedio
    assert f(a)*f(b) < 0 , "El intervalo [a, b] no contiene raíces"

    m = (a + b)/2                    # punto medio inicial
    for k in range(max_iter):        # ejecutamos loop hasta max_iter
        abs_error = np.abs(b - a)    # ancho del intervalo

        # imprimimos el intervalo y el error
        print(f'k = {k:2d}, (a,b) = ({a:.3e}, {b:.3e}), abs_error = {abs_error:.3e}')

        # si el error es menor que la tolerancia, terminamos el loop
        if abs_error < abs_tol:
            break

        m = (a + b)/2                # punto medio entre [a, b]

        # seleccionamos el nuevo intervalo según el teorema del valor intermedio
        if f(a)*f(m) < 0 : b = m     # la raíz está en [a, m]
        else             : a = m     # la raíz está en [m, b]

    # si la solución no converge y k > max_iter, informar a usuario
    if abs_error > abs_tol : print('La solución no converge')

    return m

In [3]:
f = lambda x: np.exp(x) - x**2 # función de búsqueda de raíz
a, b = -1, 1                   # intervalo [a,b]

# Mostramos el valor de f en los extremos [a, b] (solo para confirmación)
print('Análisis de intervalos')
print(f'a = {a:.3f}, f(a) = {f(a):.3f}')
print(f'b = {b:.3f}, f(b) = {f(b):.3f}')

print('\nResultado método de Bisección')
x_sol = bisection(f, bracket = (a, b))
print(f'x_sol = {x_sol:.5f}, f(x_sol) = {f(x_sol):.3e}')

Análisis de intervalos
a = -1.000, f(a) = -0.632
b = 1.000, f(b) = 1.718

Resultado método de Bisección
k =  0, (a,b) = (-1.000e+00, 1.000e+00), abs_error = 2.000e+00
k =  1, (a,b) = (-1.000e+00, 0.000e+00), abs_error = 1.000e+00
k =  2, (a,b) = (-1.000e+00, -5.000e-01), abs_error = 5.000e-01
k =  3, (a,b) = (-7.500e-01, -5.000e-01), abs_error = 2.500e-01
k =  4, (a,b) = (-7.500e-01, -6.250e-01), abs_error = 1.250e-01
k =  5, (a,b) = (-7.500e-01, -6.875e-01), abs_error = 6.250e-02
k =  6, (a,b) = (-7.188e-01, -6.875e-01), abs_error = 3.125e-02
k =  7, (a,b) = (-7.188e-01, -7.031e-01), abs_error = 1.562e-02
k =  8, (a,b) = (-7.109e-01, -7.031e-01), abs_error = 7.812e-03
k =  9, (a,b) = (-7.070e-01, -7.031e-01), abs_error = 3.906e-03
k = 10, (a,b) = (-7.051e-01, -7.031e-01), abs_error = 1.953e-03
k = 11, (a,b) = (-7.041e-01, -7.031e-01), abs_error = 9.766e-04
k = 12, (a,b) = (-7.036e-01, -7.031e-01), abs_error = 4.883e-04
k = 13, (a,b) = (-7.036e-01, -7.034e-01), abs_error = 2.441e-04
k 

#### ¿Cuántas iteraciones cuesta?

Cada paso reduce el intervalo a la mitad, así que después de $n$ pasos su ancho es $(b-a)/2^n$. El número de iteraciones para una tolerancia dada se conoce **antes** de empezar:

\begin{equation*}
n = \left\lceil \log_2\left(\frac{b-a}{\mathrm{tol}}\right) \right\rceil
\end{equation*}

> **Importante.** Esta cota es única para bisección, **ningún otro método de búsqueda de raíces tiene esta propiedad.** La fórmula permite predecir su costo computacional. A cambio avanza un dígito binario por iteración, lo que llamamos convergencia **lineal**. 

### Método de Newton-Raphson (intervalo abierto)

Es el método de intervalo abierto más usado, y sale directamente de la [aproximación lineal de la Unidad 5](../05-Taylor-series/05-Taylor-series.ipynb).

Supongamos que $x_0$ es un punto cercano a la raíz de $f(x)$. Truncando la serie de Taylor en primer orden alrededor de $x_0$, y pidiendo que esa recta se anule en $x_1$:

\begin{equation*}
0 = f(x_0) + f^{\prime}(x_0)(x_1-x_0)
\end{equation*}

Es decir, la raíz de $f(x)$ está dada por:

\begin{equation*}
x_1 = x_0 - \frac{f(x_0)}{f^{\prime}(x_0)}
\end{equation*}

Si $x_1$ no es la raíz, podemos encontrar un nuevo valor mediante $x_2 = x_1 - \frac{f(x_1)}{f^{\prime}(x_1)}$

En resumen, el método de Newton-Raphson se define mediante la operación iterativa:

\begin{equation*}
x_{k+1} = x_k - \frac{f(x_k)}{f^{\prime}(x_k)}
\end{equation*}

Gráficamente, lo que hacemos en cada iteración es encontrar el punto $x_{k+1}$ donde la recta $f(x_k) + f^{\prime}(x_k)(x-x_k)$ intersecta el eje $y = 0$.  

<img src="./images/newton_raphson.png" width="380" align= center>

La ventaja de este algoritmo es que, a diferencia de los métodos por intervalo acotado, solo necesita de un valor inicial. Esta es una característica general de los métodos de intervalo abierto.

La segunda ventaja es la velocidad: cerca de la raíz, cada iteración **duplica** el número de cifras correctas. Lo veremos con números más abajo.

La siguiente función implementa el método de Newton-Raphson

In [4]:
import numpy as np
def newton_raphson(x0, f, fprime, abs_tol = 1E-5, max_iter = 100):
    '''
    Método de Newton-Raphson para encontrar la raíz de f(x)

    Parameters:
        - x0: float
            valor inicial

        - f: callable
            función de búsqueda de raíz f(x)

        - fprime: callable
            derivada de la función de búsqueda df/dx

        - abs_tol: float (optional)
            tolerancia al valor absoluto (1E-5 por defecto)

        - max_iter: int (optional)
            número máximo de iteraciones (100 por defecto)

    Return:
        x_sol: float
            raíz de f(x)

    '''
    xk  = x0            # valor inicial
    xk1 = x0            # inicializamos el siguiente iterado
    abs_error = np.inf  # inicializamos el error absoluto

    # comenzamos las iteraciones
    for k in range(max_iter):

        # newton-raphson
        xk1 = xk - f(xk)/fprime(xk) # iteración k
        abs_error = abs(xk1 - xk)   # error absoluto

        # imprimimos xk1 y el error
        print(f'k = {k:d}, xk1 = {xk1:.5e}, abs_error = {abs_error:.3e}')

        # si el error es menor a la tolerancia, terminamos el loop
        if abs_error < abs_tol:
            break

        # si no, actualizamos xk
        xk = xk1

    # si k > max_iter y la solución no converge, indicar al usuario
    if abs_error > abs_tol : print('La solución no converge')

    return xk1

Probamos el método para encontrar la raíz de $f(x) = e^{x} - x^2$ con $f'(x) = e^{x} - 2x$.

In [5]:
f    = lambda x: np.exp(x) - x**2
dfdx = lambda x: np.exp(x) - 2*x

x0 = 1 # valor inicial
x_sol = newton_raphson(x0, f, dfdx)
print(f'x_sol = {x_sol:.5f}, f(x_sol) = {f(x_sol):.3e}')

k = 0, xk1 = -1.39221e+00, abs_error = 2.392e+00
k = 1, xk1 = -8.35088e-01, abs_error = 5.571e-01
k = 2, xk1 = -7.09834e-01, abs_error = 1.253e-01
k = 3, xk1 = -7.03483e-01, abs_error = 6.351e-03
k = 4, xk1 = -7.03467e-01, abs_error = 1.598e-05
k = 5, xk1 = -7.03467e-01, abs_error = 1.011e-10
x_sol = -0.70347, f(x_sol) = 0.000e+00


Newton llega a la misma raíz $x^* = -0{,}70347$ en 6 iteraciones, contra 18 de la bisección con la misma tolerancia. Notemos además que el primer paso lo lleva a $-1.39$, fuera del intervalo $[-1,1]$: un método abierto no respeta ninguna cota.

#### Caso $f'(x)$ no es analítica (Método de la secante)

Otros métodos de intervalo abierto se diferencian de Newton-Raphson en la forma de determinar $f'(x)$. Esto debido a que no siempre es posible determinar la derviada de forma analítica.

Por ejemplo, en el **método de la secante**, aproxima $f'(x)$ por $f'(x_k) = \frac{f(x_k) - f(x_{k-1})}{x_k - x_{k-1}}$, lo que deriva en la siguiente fórmula recursiva:

\begin{equation*}
x_{k+1} = x_k - \frac{f(x_k)(x_k - x_{k-1})}{f(x_k) - f(x_{k-1})}
\end{equation*}

Notar que, debido a esta fórmula, el método de la secante requiere **dos valores iniciales, $x_0$ y $x_1$**

> Reemplazar $f'(x_k)$ por $\frac{f(x_k)-f(x_{k-1})}{x_k-x_{k-1}}$ es aproximar una derivada por una diferencia dividida, y arrastra el mismo compromiso entre truncamiento y redondeo que vimos en la **Unidad 5**.

#### Análisis de convergencia 

En la siguiente animación, elijamos una función, movamos el valor inicial $x_0$ o los extremos del intervalo, e iteremos paso a paso. Observemos la columna del error: cuántos ceros gana cada método en cada iteración.

In [6]:
HTML(Path('interactive/A3_biseccion_vs_newton.html').read_text(encoding='utf-8'))

k,cota bisec.,error Newton


De la animación desprendemos las siguientes **conclusiones sobre los métodos de intervalo abierto**:

- **La convergencia es significativamente más rápida que métodos de intervalo cerrado.**  Sobre la función de Colebrook, partiendo de $x_0 = 0{,}02$, el error de Newton pasa de $1{,}4\times10^{-3}$ a $8{,}1\times10^{-5}$, luego a $2{,}7\times10^{-7}$ y después a $2{,}8\times10^{-12}$. La bisección sobre el mismo problema necesita 23 iteraciones para llegar a esa tolerancia.

- **Están expuesto a serios  problemas de convergencia si el valor $x_k$ cae en un punto de la función donde $f'(x_k) \approx 0$**

- **No hay  control sobre la raíz encontrada.** Esto lo vemos, por ejemplo, en la función $f(x) = x^3 - 100x^2 - x + 100$, que tiene tres raíces $x^* = 1$, $x^* = -1$ y $x^* = 100$. Según lo que muestra la animación:
    - Valor inicial $x_0 = 0$, la función converge a la raíz $x^* = 100$.
    - Valor inicial $x_0 = 0.01$, la función converge a la raíz $x^* = 1$.
    - Valor inicial $x_0 < 0$, la función converge a la raíz $x^* = -1$.

### Métodos combinados
Los métodos más sofisticados para búsqueda de raíces combinan métodos de intervalo abierto y cerrado. Por un lado, el método de intervalo abierto permite una convergencia más rápida, mientras que el método de intervalo cerrado permite acotar la solución. 

En términos generales, los métodos combinados operan de la siguiente forma.
- Se subdivide el dominio de la función para identificar intervalos donde existan raíces.
- Se procede con la iteración mediante un método de intervalo abierto
- Si la solución se mueve fuera del intervalo acotado, se procede a iterar con un método de intervalo cerrado.

Por ejemplo, el **método de Brent** combina un método de intervalo abierto (como Newton-Raphson o secante), con el método de la bisección. Más información en las referencias

### Raíces de función escalar en python 
En python, la función ```root_scalar``` de la librería ```scipy.optimize```, permite determinar raíces de una función escalar.

Los argumento más relevantes en esta función son:
```python
from scipy.optimize import root_scalar
root_scalar(f,            # callable, función objetivo
            args=(),      # tuple (opcional), argumentos extra para la función
            method=None,  # str (opcional), tipo de método
            bracket=None, # 2 floats list (opcional), intervalo de búsqueda de raíces
            fprime=None,  # callable (opcional), primera derivada
            x0=None,      # float (opcional), valor inicial
            x1=None,      # float (opcional), 2do valor inicial (para aproximar derivada)
            xtol=None,    # float (opcional), tolerancia para error absoluto
            rtol=None,    # float (opcional), tolerancia para error relativo
            maxiter=None  # int (opcional), número máximo de operaciones
           )
```

La función tiene implementada distintos métodos de intervalo abierto, cerrado y combinados, tales como: bisección (`bisect`), Newton-Raphson (`newton`), secante (`secant`) y Brent's (`brentq` o `brenth`). El tipo de método debe ir indicado en `method`:

Además de la función objetivo, se debe indicar un valor inicial, dos valores iniciales o un intervalo. Con esto `root_scalar` definirá el tipo de método dependiendo del input: 

- Un valor inicial $x_0$ (variable `x0`), y la derivada (variable `fprime`) $\rightarrow$ método de intervalo abierto o híbrido
- Dos valores iniciales $x_0$ (variable `x0`) y $x_1$ (variable `x1`) $\rightarrow$ método de intervalo abierto o híbrido
- Un intervalo. (variable `bracket`)  $\rightarrow$ método de intervalo cerrado o híbrido

El tipo de argumento que pide cada método está en la siguiente tabla (x = requerido, o = opcional):

|`method`| `f` |`args`|`bracket`| `x0` | `x1` |`fprime`|`xtol`|`rtol`|`maxiter`|
|:----:|:-:|:--:|:------:|:--:|:--:|:----:|:--:|:--:|:-----:|
|`'bisect'`|x|o|x| | | |o|o|o|
|`'brentq'`|x|o|x| | | |o|o|o|
|`'brenth'`|x|o|x| | | |o|o|o|
|`'secant'`|x|o| |x|x| |o|o|o|
|`'newton'`|x|o| |x| |x|o|o|o|

Por ejemplo, analicemos la raíz de la función $f(x) = x^3 - 1$.

In [7]:
from scipy.optimize import root_scalar
f  = lambda x: x**3 - 1 # función objetivo f(x)
df = lambda x: 3*x**2   # primera derivada de f(x)

Usamos `root_scalar` con el argumento `method` para especificar el tipo de método que queremos utilizar

In [8]:
print('Bisección:\n',      root_scalar(f,bracket=[0, 3]  ,method='bisect'))
print('\nNewton-Raphson:\n', root_scalar(f,x0=0.2,fprime=df,method='newton'))
print('\nSecante:\n',        root_scalar(f,x0=0.2,x1=0.21  ,method='secant'))

Bisección:
       converged: True
           flag: converged
 function_calls: 43
     iterations: 41
           root: 1.0000000000004547
         method: bisect

Newton-Raphson:
       converged: True
           flag: converged
 function_calls: 22
     iterations: 11
           root: 1.0
         method: newton

Secante:
       converged: True
           flag: converged
 function_calls: 23
     iterations: 22
           root: 1.0
         method: secant


Si omitimos el argumento `method`, `root_scalar` utilizará el método más adecuado en base al tipo de input (`x0`, `bracket`, `fprime`, ect) 

In [9]:
print('Brent´s:\n',                                         root_scalar(f,bracket=[0, 3],method='brentq'))
print('\nIntervalo (método por defecto):\n',                root_scalar(f,bracket=[0, 3]))
print('\nCond. inicial y derivada (método por defecto):\n', root_scalar(f,x0=0.2,fprime=df))

Brent´s:
       converged: True
           flag: converged
 function_calls: 11
     iterations: 10
           root: 1.0
         method: brentq

Intervalo (método por defecto):
       converged: True
           flag: converged
 function_calls: 11
     iterations: 10
           root: 1.0
         method: brentq

Cond. inicial y derivada (método por defecto):
       converged: True
           flag: converged
 function_calls: 22
     iterations: 11
           root: 1.0
         method: newton


El resultado de `root_scalar` es un objeto que guarda la raíz junto con el diagnóstico de la iteración. El valor se extrae con `.root`.

In [10]:
sol = root_scalar(f, bracket=[0, 3])
print(f'raíz x* = {sol.root:.5f}, en {sol.iterations} iteraciones')

raíz x* = 1.00000, en 10 iteraciones


Las tolerancias y el tope de iteraciones se controlan con tres argumentos:

- `xtol` para el error absoluto: `root_scalar(f, bracket=[0,3], xtol=1E-5)`
- `rtol` para el error relativo: `root_scalar(f, bracket=[0,3], rtol=0.001)`
- `maxiter` para el número máximo de iteraciones

Si damos `xtol` y `rtol` a la vez, la iteración termina apenas se cumpla cualquiera de los dos.

Para mayor información revisar la [documentación oficial](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html).

#### El problema de Colebrook

Volvamos al problema con el que abrimos el capítulo: agua a 20 °C ($\nu = 1{,}004\times10^{-6}$ m²/s, $\rho = 998$ kg/m³) por una tubería de acero comercial ($\varepsilon = 0{,}045$ mm), con $D = 0{,}1$ m, $L = 100$ m y $V = 2$ m/s.

In [11]:
nu, rho, g = 1.004e-6, 998.0, 9.81  # agua a 20 °C
D, L, V    = 0.1, 100.0, 2.0        # geometría y velocidad
eps        = 0.045e-3               # rugosidad del acero comercial

Re   = V*D/nu                       # número de Reynolds
epsD = eps/D                        # rugosidad relativa

# función de búsqueda de raíz: Colebrook igualada a cero
f = lambda fc: 1/np.sqrt(fc) + 2.0*np.log10(epsD/3.7 + 2.51/(Re*np.sqrt(fc)))

print(f'Re = {Re:.3e}, rugosidad relativa = {epsD:.2e}')

Re = 1.992e+05, rugosidad relativa = 4.50e-04


El factor de fricción de una tubería siempre cae entre 0.008 y 0.08 (esto es dato), de modo que tenemos un intervalo natural para un método cerrado.

In [12]:
sol = root_scalar(f, bracket=[0.008, 0.08])   # Brent por defecto
fc  = sol.root
print(f'f_c = {fc:.6f}  ({sol.iterations} iteraciones)')

f_c = 0.018567  (11 iteraciones)


## Búsqueda de raíces para funciones vectoriales
Una función vectorial es una función con una o más variables independientes, que entrega un vector de múltiples dimensiones.

\begin{equation*}
f: x\in \mathbb{R}^n \rightarrow \mathbb{R}^m
\end{equation*}

Consideremos el siguiente sistema de ecuaciones:
\begin{align*}
x\log(y^2-1) &= 3 \\
y\sin(2x^3) + e^y &= 2
\end{align*}

Resolver este sistema equivale a encontrar las raíces de una función vectorial:

\begin{equation*}
\mathbf{F}(x,y) = \begin{cases}
f(x,y) = x\log(y^2-1) - 3 \\
g(x,y) = y\sin(2x^3) + e^y - 2
\end{cases}
\end{equation*}

Así, resolver un sistema de ecuaciones de $n$ incognitas, es equivalente a encontrar las raíces de una función vectorial del tipo:

\begin{equation*}
f: x\in \mathbb{R}^n \rightarrow \mathbb{R}^n
\end{equation*}

En este capítulo revisaremos los aspectos generales de los métodos numéricos para resolver este problema.

### Newton-Raphson en varias dimensiones

Para un vector $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ y una función vectorial $\mathbf{F}(\mathbf{x}) = \big(f_1(\mathbf{x}), f_2(\mathbf{x}), \ldots, f_n(\mathbf{x})\big)$, la forma generalizada de Newton-Raphson es:

\begin{equation*}
J(\mathbf{x}_k)\,\Delta\mathbf{x} = -\,\mathbf{F}(\mathbf{x}_k),
\qquad \mathbf{x}_{k+1} = \mathbf{x}_k + \Delta\mathbf{x}
\end{equation*}

donde $J = \nabla\mathbf{F}$ es el **Jacobiano** de $\mathbf{F}$.

> **Nota.** Cada iteración de Newton vectorial **es un sistema lineal** $A\mathbf{x} = \mathbf{b}$ de los de la [**Unidad 2**](../02-Algebra_lineal/02-Algebra_lineal.ipynb), con $A = J$ y $\mathbf{b} = -\mathbf{F}$. Por eso se resuelve con `solve` y no invirtiendo $J$: invertir cuesta más operaciones y pierde precisión.

#### El Jacoviano como estimador de avance

 El **operador $\nabla \mathbf{F}$ corresponde a una matriz**, donde cada elemento está dado por la derivada parcial de una componente de la función respecto a un parámetro independiente, es decir: $J_{ij} = \frac{\partial f_i}{\partial x_j}$

Por ejemplo, para una función vectorial $\mathbf{F}(x,y) = \big(f(x,y),\, g(x,y)\big)$, el Jacobiano está dado por:

\begin{equation*}
J(x,y) =
\begin{bmatrix}
\frac{\partial f}{\partial x} & \frac{\partial f}{\partial y} \\
\frac{\partial g}{\partial x} & \frac{\partial g}{\partial y}
\end{bmatrix}
\end{equation*}

En otras palabras, el Jacobiano es equivalente a la derivada pero para funciones vectoriales.


El método generalizado de Newton-Raphson, así, consiste en encontrar un nuevo vector $\mathbf{x}_{k+1}$ a partir de la pendiente descendiente definida en el vector $\mathbf{x}_{k}$.

#### Criterio de minimización

A diferencia del caso unidimensional, el Jacobiano entrega multiples direcciones posibles. ¿Cómo saber cuál es la dirección que minimiza $\mathbf{F}$?

Para definir la dirección descendiente se considera el criterio:

\begin{equation*}
\mathrm{min}\left[\mathbf{F}\cdot\mathbf{F}\right] 
\end{equation*}

Así, el problema de busqueda de raíces de una función vectorial se transforma en un problema de minimización.

Esta es la estrategia de los **métodos de búsqueda lineal**.

Entre los más conocidos tenemos el **método de Broyden**. Más información en las referencias

### Métodos de región de confianza

En general, determinar el Jacobiano de una función vectorial es complicado. A raíz de esto nacen los métodos de región de confianza, los cuales se basan en una aproximación de $\mathbf{F}$ en forma de paraboloide. Esta aproximación simplifica el cálculo del Jacobiano. 

Se define como **región de confianza a la región donde la función puede ser aproximada por un parabolide**.

En términos generales, los métodos de región de confianza operan de la siguiente forma:

- Se define una región de confianza inicial y se busca un mínimo dentro esa región.
- Si el valor encontrado minimiza $\mathbf{F}\cdot\mathbf{F}$, se construye una aproximación paraboloide de $\mathbf{F}$ y se incrementa la región de confianza.
- Si el valor encontrado no minimiza $\mathbf{F}\cdot\mathbf{F}$, se reduce la región de confianza, y se vuelve a buscar el mínimo.
- El algoritmo itera hasta encontrar un mínimo global de $\mathbf{F}$.

En la siguiente animación seguimos el algoritmo paso a paso sobre un sistema de dos ecuaciones con un valle curvo. Observemos el paso de Newton completo, en azul, contra el paso que la región permite, en rojo.

In [ ]:
HTML(Path('interactive/A4_region_de_confianza.html').read_text(encoding='utf-8'))

> La decisión la toma $\rho$, el cociente entre la reducción real de $\mathbf{F}\cdot\mathbf{F}$ y la que el modelo había prometido. Si $\rho$ es alto el modelo sirve, el paso se acepta y el radio crece. Si $\rho$ es bajo o negativo el paso se descarta y el radio se reduce a la mitad, sin moverse del punto.

> Partiendo de $(-0.5,\,-0.4)$ con un radio de 2, los dos primeros pasos se rechazan y el radio baja de 2 a 0.5 antes de que el primero sea aceptado. Ese es el trabajo del método: descubrir cuánto se le puede creer al modelo en cada punto.

En general, los métodos de región de confianza son más estables que los métodos de búsqueda lineal, y son los métodos por defecto en funciones de python.

Mayor información sobre estos métodos [acá](http://www.applied-mathematics.net/optimization/optimizationIntro.html)

### Raíces de función vectorial en python
En python, la función ```fsolve``` de la librería ```scipy.optimize``` permite encontrar las raíces de una función vectorial.

Los principales inputs de la función son:

```python
scipy.optimize.fsolve(func,             # función vectorial objetivo (callable)
                      x0,               # valores iniciales
                      args=(),          # argumentos extras para la iteración
                      xtol=1.49012e-08  # tolerancia al error relativo
                     )
```

La función se basa en los algoritmos de región de confianza "hybrd" y "hybrj" de la libreria ```MINPACK```. Más detalles [acá](https://www.math.utah.edu/software/minpack/minpack/hybrj.html)

La función ```fsolve``` requiere, como mínimo, la función vectorial y los valores iniciales.

Por ejemplo, queremos resolver el sistema

\begin{align*}
x\cos(y)=4 \\
xy-y=5
\end{align*}

In [13]:
import numpy as np
from scipy.optimize import fsolve
def func(xvec):
    # Usamos el vector de entrada para definir cada variable de la ecuación (RECOMENDADO)
    x, y = xvec[0], xvec[1]

    # Retornamos el output en formato numpy array (RECOMENDADO)
    return np.array([
        x*np.cos(y) - 4,
        x*y - y - 5    ])

root = fsolve(func, x0 = [1, 1])
print(f'la solución es: x = {root[0]:.5f}, y = {root[1]:.5f}')

la solución es: x = 6.50410, y = 0.90841


In [14]:
func(root)

array([3.73212572e-12, 1.61701763e-11])

También podemos definir el error absoluto mediante la instrucción ```xtol``` (por defecto, ```xtol=1.49012e-08```).
```python
root = fsolve(func, [1, 1], xtol = 1E-10) # |xk+1 - xk| < 1E-10
```

Para mayor información, revisar la [documentación oficial](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fsolve.html)

## Resumen: ¿qué método uso?

| Situación | Método | Costo | En Python |
|---|---|---|---|
| tengo un intervalo con cambio de signo y quiero garantía | bisección | $\lceil\log_2\frac{b-a}{\mathrm{tol}}\rceil$ iteraciones | `root_scalar(f, bracket=[a,b], method='bisect')` |
| tengo la derivada y un buen valor inicial | Newton-Raphson | cuadrático, 2 evaluaciones por paso | `root_scalar(f, x0=.., fprime=..)` |
| no tengo la derivada | secante | superlineal, 1 evaluación por paso | `root_scalar(f, x0=.., x1=..)` |
| quiero velocidad sin perder la garantía | Brent | cuadrático con respaldo de bisección | `root_scalar(f, bracket=[a,b])` |
| el problema tiene varias incógnitas | Newton vectorial, región de confianza | un sistema lineal por iteración | `scipy.optimize.fsolve` |

Antes de confiar en una raíz conviene revisar siempre:

1. ¿Graficamos $f(x)$? Casi todos los fracasos de un método abierto se ven venir en el gráfico.
2. ¿$f(x^*)$ es realmente pequeño? Un $x_k$ que dejó de moverse no siempre es una raíz.
3. ¿La función tiene más de una raíz? Un método abierto entrega la que encuentre, no la que buscamos.
4. ¿El resultado tiene sentido físico? Un factor de fricción negativo es una raíz matemática, no una solución.

## Referencias

- Chapra S. **Chapter 5: Roots — Bracketing Methods** y **Chapter 6: Roots — Open Methods** en *Applied Numerical Methods with MATLAB for Engineers*, 3rd Ed., McGraw Hill.
  - §5.3 acotamiento y valores iniciales · §5.4 bisección y su número de iteraciones · §6.2 Newton-Raphson · §6.3 secante · §6.4 método de Brent · §6.7 caso de estudio: fricción en tuberías

- Press W., Teukolsky S., Vetterling W., Flannery B. **Chapter 9: Root Finding and Nonlinear Sets of Equations** en *Numerical Recipes: The Art of Scientific Computing*, 3rd Ed., Cambridge University Press, 2007.
  - §9.1 acotamiento y bisección · §9.2 secante y falsa posición · §9.3 método de Brent · §9.4 Newton-Raphson con derivada · §9.6 Newton para sistemas no lineales · §9.7 métodos globalmente convergentes

- Chapra S., Canale R. **Parte dos: Raíces de ecuaciones** en *Métodos Numéricos para Ingenieros*, 6ta Ed., McGraw Hill, 2011

- Kong Q., Siauw T., Bayen A. M. **Chapter 19: Root Finding** en *[Python Programming and Numerical Methods – A Guide for Engineers and Scientists](https://pythonnumericalmethods.berkeley.edu/notebooks/chapter19.00-Root-Finding.html)*, 1st Ed., Academic Press, 2021